# Tutorial 02: Detección de Actividad Muscular sEMG y Poda de Pesos (Pruning)
### TinyML para Raspberry Pi Pico 2 / Pico 2 W (RP2350) con TensorFlow Model Optimization (TFMOT)

En este tutorial aprenderás:
1. **Generación de bioseñales sEMG** (Electromiografía superficial) para 3 estados de activación: Reposo, Contracción Ligera y Contracción Fuerte.
2. **Concepto de Poda de Pesos (Weight Pruning / Sparsity)**: eliminar sinapsis redundantes forzando pesos menores a cero mediante un programa de decaimiento polinomial:
   $$s_t = s_f + (s_i - s_f)\left(1 - \frac{t - t_0}{n}\right)^3$$
3. **Evaluación del impacto en la compresión**: las matrices dispersas se comprimen dramáticamente con algoritmos como Gzip o Huffman, reduciendo transferencias OTA y memoria Flash.
4. **Combinación de Pruning + INT8 Quantization**.
5. **Despliegue e inferencia en Arduino IDE para la Raspberry Pi Pico 2** con respuesta visual en LED.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.signal_generator import generate_emg_dataset
from utils.exporter import tflite_to_c_header, save_test_samples_header
from utils.tflite_eval import evaluate_tflite_model

## 1. Generación y Visualización de Señales sEMG con NeuroKit2

In [ ]:
CLASS_NAMES = ["Rest", "Low_Contraction", "High_Contraction"]
SAMPLE_LEN = 200
SAMPLING_RATE = 1000

X, y = generate_emg_dataset(num_samples_per_class=400, sample_len=SAMPLE_LEN, sampling_rate=SAMPLING_RATE)
X = np.expand_dims(X, axis=-1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for c_idx, name in enumerate(CLASS_NAMES):
    idx = np.where(y_train == c_idx)[0][0]
    axes[c_idx].plot(X_train[idx, :, 0], color='tab:green' if c_idx==0 else ('tab:orange' if c_idx==1 else 'tab:red'))
    axes[c_idx].set_title(f"{name}", fontweight='bold')
    axes[c_idx].grid(True, linestyle="--", alpha=0.5)
plt.suptitle("Ráfagas sEMG para Reconocimiento de Gestos/Contracción", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Definición del Modelo Base y Entrenamiento

In [ ]:
def build_emg_cnn():
    inputs = tf.keras.Input(shape=(SAMPLE_LEN, 1))
    x = tf.keras.layers.Conv1D(16, 9, strides=2, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPool1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 5, strides=2, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPool1D(2)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    outputs = tf.keras.layers.Dense(3, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

base_model = build_emg_cnn()
base_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
base_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=15, batch_size=32, verbose=1)

_, base_acc = base_model.evaluate(X_test, y_test, verbose=0)
print(f"Exactitud Modelo Base: {base_acc * 100:.2f}%")

## 3. Poda de Pesos (Weight Pruning a 65% de Dispersión)
Utilizamos `tensorflow_model_optimization` para inducir dispersión estructurada.

In [ ]:
import tensorflow_model_optimization as tfmot

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.0,
    final_sparsity=0.65,
    begin_step=0,
    end_step=int(np.ceil(len(X_train) / 32)) * 12
)
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(base_model, pruning_schedule=pruning_schedule)
pruned_model.compile(optimizer=tf.keras.optimizers.Adam(0.0005), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

pruned_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=12, batch_size=32,
                 callbacks=[tfmot.sparsity.keras.UpdatePruningStep()], verbose=1)

# Extracción del modelo limpio sin wrappers de poda
stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
_, final_acc = stripped_model.evaluate(X_test, y_test, verbose=0)
print(f"Exactitud tras 65% de Poda: {final_acc * 100:.2f}%")

## 4. Visualización del Histograma de Pesos (Efecto de Sparsity)

In [ ]:
weights_base = base_model.layers[1].get_weights()[0].flatten()
weights_pruned = stripped_model.layers[1].get_weights()[0].flatten()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(weights_base, bins=40, color="steelblue", edgecolor="black")
ax1.set_title("Pesos Modelo Base (Denso)")
ax1.set_xlabel("Valor del Peso")
ax1.grid(True, alpha=0.5)

ax2.hist(weights_pruned, bins=40, color="forestgreen", edgecolor="black")
ax2.set_title("Pesos Modelo Podado (65% Sparsity en 0)")
ax2.set_xlabel("Valor del Peso")
ax2.grid(True, alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Conversión a TFLite INT8 y Exportación a Arduino (`model_data.h`)

In [ ]:
def rep_dataset():
    for i in range(150):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_pruned_int8 = converter.convert()

ARDUINO_DIR = Path("../arduino/emg_inference_pico2")
ARDUINO_DIR.mkdir(parents=True, exist_ok=True)

tflite_to_c_header(tflite_pruned_int8, str(ARDUINO_DIR / "model_data.h"))
save_test_samples_header(X_test[:, :, 0], y_test, str(ARDUINO_DIR / "test_samples.h"), CLASS_NAMES, samples_per_class=3)
print("Archivos C de EMG exportados con éxito!")